In [ ]:
import pairadigm as pdm
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from typing import List, Tuple, Optional, Union, Dict
import numpy as np
from tqdm import tqdm
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr
from pairadigm import Pairadigm
import copy
import random
import polars as pl

import sys
from datetime import datetime
import os

In [ ]:
# Set random seeds for reproducibility
def set_seeds(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Detect best device available
def get_best_device(cuda_index: int = 0, verbose: bool = True):
    """Detect best device: CUDA (with index) > MPS > CPU"""
    if torch.cuda.is_available():
        if cuda_index < torch.cuda.device_count():
            device = torch.device(f"cuda:{cuda_index}")
            device_name = f"cuda:{cuda_index}"
            if verbose:
                try:
                    print(f"Using NVIDIA GPU {cuda_index}: {torch.cuda.get_device_name(cuda_index)}")
                except Exception:
                    print(f"Using NVIDIA GPU {cuda_index}")
        else:
            raise ValueError(f"CUDA device {cuda_index} not available. Available devices: 0 to {torch.cuda.device_count()-1}")

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        device_name = "mps"
        if verbose:
            print("Using Apple Silicon GPU via MPS")
    else:
        device = torch.device("cpu")
        device_name = "cpu"
        if verbose:
            print("No GPU found, using CPU")
    return device, device_name

# Set seeds and get device
seed_num = 42
set_seeds(seed=seed_num)
device, device_name = get_best_device(cuda_index=0, verbose=True)

In [ ]:
# Load emopair dataset (contains id, split, V, text, Aro/Dom BT scores)
print("Loading data...")
emopair = pl.read_csv("../Results/emopair.csv").to_pandas()
emopair_pairs = pl.read_csv("../Results/emopair_pairs_w_breakdowns.csv").to_pandas()

print(f"Data loaded. {len(emopair)} EmoPair items. {len(emopair_pairs)} pairs.")

In [ ]:
emopair

In [ ]:
# Split data into train/dev/test based on 'split' column in emopair_pairs
train_set_pairs = emopair_pairs[(emopair_pairs['item1_split'] == 'train') & (emopair_pairs['item2_split'] == 'train')].copy()
print(f"Training set prepared. Number of pairs: {len(train_set_pairs)}")

eval_set_pairs = emopair_pairs[(emopair_pairs['item1_split'] == 'dev') & (emopair_pairs['item2_split'] == 'dev')].copy()
print(f"Evaluation set prepared. Number of pairs: {len(eval_set_pairs)}")

test_set_pairs = emopair_pairs[(emopair_pairs['item1_split'] == 'test') & (emopair_pairs['item2_split'] == 'test')].copy()
print(f"Test set prepared. Number of pairs: {len(test_set_pairs)}")

In [ ]:
emopair_pairs